# Actividad 06 – Ítem D: Detección de objetos con YOLO en video (simulación de tiempo real)
**Visión Artificial – UNA Puno – X ciclo, grupo B**

Este notebook procesa un video **cuadro por cuadro** (como si fuera un stream en vivo), detecta objetos con YOLO, guarda el video anotado y registra métricas por frame (FPS, latencia, nº de detecciones, confianza) para usarlas en el **ítem E** (comparación imágenes vs. video y RCNN vs. YOLO).

> Ejecuta con GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.

## 1. Instalación

In [ ]:
!pip -q install ultralytics

## 2. Configuración
Cambia estos parámetros. **Usen los mismos videos y el mismo `CONF`/`IMGSZ` que el equipo de Faster-RCNN** para que la comparación sea justa.

In [ ]:
import os, time, cv2, numpy as np, pandas as pd, torch
from ultralytics import YOLO

MODEL      = "yolo11n.pt"   # prueba también yolo11s.pt / yolo11m.pt
CONF       = 0.25           # umbral de confianza
IMGSZ      = 640            # tamaño de entrada
MAX_FRAMES = 300            # None = todo el video
VIDEO_PATH = "video_entrada.mp4"

device = 0 if torch.cuda.is_available() else "cpu"
print("Dispositivo:", "GPU " + torch.cuda.get_device_name(0) if device == 0 else "CPU")

## 3. Cargar el video
Opción A: sube tu propio video. Opción B: descarga uno de ejemplo.

In [ ]:
from google.colab import files

USAR_VIDEO_PROPIO = True   # False = descargar video de ejemplo

if USAR_VIDEO_PROPIO:
    up = files.upload()
    nombre = list(up.keys())[0]
    os.replace(nombre, VIDEO_PATH)
else:
    !wget -q -O video_entrada.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle-car-detection.mp4

cap = cv2.VideoCapture(VIDEO_PATH)
print("Resolución:", int(cap.get(3)), "x", int(cap.get(4)),
      "| FPS original:", cap.get(cv2.CAP_PROP_FPS),
      "| Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
cap.release()

## 4. Detección frame a frame (simulación de tiempo real)

In [ ]:
model = YOLO(MODEL)
names = model.names

cap = cv2.VideoCapture(VIDEO_PATH)
fps_in = cap.get(cv2.CAP_PROP_FPS) or 30
W, H = int(cap.get(3)), int(cap.get(4))
out = cv2.VideoWriter("salida_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps_in, (W, H))

# warm-up: el primer frame siempre es más lento y distorsiona el FPS
model.predict(np.zeros((H, W, 3), np.uint8), imgsz=IMGSZ, device=device, verbose=False)

rows, class_counts = [], {}
idx = 0
t_global = time.perf_counter()
while True:
    ok, frame = cap.read()
    if not ok or (MAX_FRAMES and idx >= MAX_FRAMES):
        break
    t0 = time.perf_counter()
    r = model.predict(frame, conf=CONF, imgsz=IMGSZ, device=device, verbose=False)[0]
    dt = time.perf_counter() - t0                      # latencia de extremo a extremo (s)

    b = r.boxes
    confs = b.conf.cpu().numpy() if len(b) else np.array([])
    clss  = b.cls.cpu().numpy().astype(int) if len(b) else np.array([])
    for c in clss:
        class_counts[names[c]] = class_counts.get(names[c], 0) + 1

    ann = r.plot()
    cv2.putText(ann, f"{1/dt:.1f} FPS | {len(b)} obj", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    out.write(ann)

    rows.append({"frame": idx, "latencia_ms": dt*1000, "fps": 1/dt,
                 "n_detecciones": len(b),
                 "conf_media": float(confs.mean()) if len(confs) else np.nan,
                 "inf_ms": r.speed["inference"]})
    idx += 1

cap.release(); out.release()
total = time.perf_counter() - t_global
df = pd.DataFrame(rows)
print(f"Procesados {idx} frames en {total:.1f}s")

## 5. Ver el video resultante

In [ ]:
!ffmpeg -y -loglevel error -i salida_raw.mp4 -vcodec libx264 -pix_fmt yuv420p salida_yolo.mp4

from IPython.display import HTML
from base64 import b64encode
data = b64encode(open("salida_yolo.mp4", "rb").read()).decode()
HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## 6. Métricas (insumo para el ítem E)

In [ ]:
import matplotlib.pyplot as plt

resumen = {
    "modelo": MODEL, "imgsz": IMGSZ, "conf": CONF, "dispositivo": str(device),
    "frames": len(df),
    "FPS_promedio": round(df.fps.mean(), 2),
    "FPS_mediana": round(df.fps.median(), 2),
    "latencia_media_ms": round(df.latencia_ms.mean(), 2),
    "latencia_p95_ms": round(df.latencia_ms.quantile(0.95), 2),
    "detecciones_por_frame": round(df.n_detecciones.mean(), 2),
    "confianza_media": round(df.conf_media.mean(), 3),
}
print(pd.Series(resumen).to_string())

fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
ax[0].plot(df.fps);            ax[0].set_title("FPS por frame")
ax[1].plot(df.latencia_ms);    ax[1].set_title("Latencia (ms)")
ax[2].plot(df.n_detecciones);  ax[2].set_title("Detecciones por frame")
for a in ax: a.set_xlabel("frame")
plt.tight_layout(); plt.savefig("metricas_video_yolo.png", dpi=150); plt.show()

pd.Series(class_counts).sort_values(ascending=False).head(10).plot.barh(title="Clases detectadas (top 10)")
plt.gca().invert_yaxis(); plt.show()

df.to_csv("metricas_video_yolo.csv", index=False)
pd.Series(resumen).to_json("resumen_video_yolo.json")

## 7. (Opcional) mAP en imágenes – para comparar imágenes vs. video en el ítem E
En video no hay *ground truth*, así que solo medimos velocidad y estabilidad. La **precisión (mAP)** se mide en imágenes con anotaciones. Aquí usamos COCO128 (pequeño, rápido).

In [ ]:
m = model.val(data="coco128.yaml", imgsz=IMGSZ, device=device, verbose=False)
print(f"mAP50: {m.box.map50:.3f} | mAP50-95: {m.box.map:.3f} | "
      f"Precision: {m.box.mp:.3f} | Recall: {m.box.mr:.3f}")

## 8. (Opcional) Comparar tamaños de modelo
Útil para mostrar el compromiso velocidad vs. precisión.

In [ ]:
comp = []
for mn in ["yolo11n.pt", "yolo11s.pt"]:
    mdl = YOLO(mn)
    mdl.predict(np.zeros((H, W, 3), np.uint8), imgsz=IMGSZ, device=device, verbose=False)
    cap = cv2.VideoCapture(VIDEO_PATH); ts = []; n = 0
    while n < 100:
        ok, f = cap.read()
        if not ok: break
        t0 = time.perf_counter()
        mdl.predict(f, conf=CONF, imgsz=IMGSZ, device=device, verbose=False)
        ts.append(time.perf_counter() - t0); n += 1
    cap.release()
    comp.append({"modelo": mn, "FPS": round(1/np.mean(ts), 1), "latencia_ms": round(np.mean(ts)*1000, 1)})
pd.DataFrame(comp)

## 9. Descargar resultados

In [ ]:
for f in ["salida_yolo.mp4", "metricas_video_yolo.csv", "metricas_video_yolo.png", "resumen_video_yolo.json"]:
    files.download(f)